### 1. Import libraries


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score,
                             precision_score,
                             recall_score,
                             f1_score)

### 2. Load data


In [2]:
carpeta_de_datos = "datasets/"
nombre_del_archivo = "users_behavior.csv"
data = pd.read_csv(os.path.join(carpeta_de_datos, nombre_del_archivo))

### 3. Explore data

In [3]:
data.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


Al ver los datos, entiende rápidamente que este es un problema de clasificación. Por lo tanto se propone:

Objetivo general:

- Predecir si un usuario es un usuario "SMART" o "ULTRA"; en función de la cantidad de llamadas, minutos usados, mensajes enviados y mb de navegación utilizados.

Objetivos específicos:

- Evaluar el balance de los datos (sesgos)
- Realizar una prueba de cordura para cada modelo.
- Entrenar un modelo de clasificación logística (Logistic Regression).
- Entrenar un modelo de árbol de decisión (Decision Tree).
- Entrenar un modelo de bosque aleatorio (Random Forest).
- Calcular el porcentaje de aciertos y la precisión.
- Entrenar cada modelo con diferentes hiperparámetros y comparar los resultados.
    

### 4. Análisis Machine Learning Supervisado

In [5]:
data.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


Se observan que los datos tienen diferentes órdenes de magnitud:

decenas en calls y mensajes

centenas en minutes

miles en megas usados

Esto afectará el rendimiento del modelo Logistico, se recomienda estandarizar los datos; para eso se usará StandardScaler

In [6]:
# Evaluar los datos
cantidad_ultra = data[data["is_ultra"]== 1]["is_ultra"].count()
cantidad_smart = data[data["is_ultra"]== 0]["is_ultra"].count()
print(f"Cantidad de usuarios Ultra: {cantidad_ultra}")
print(f"Cantidad de usuarios Smart: {cantidad_smart}")


Cantidad de usuarios Ultra: 985
Cantidad de usuarios Smart: 2229


Se parecia una relación 30/70 entre los usuarios Ultra y Smart, lo que podría afectar el rendimiento del modelo Logistico. Se utilizará class_weight="balanced" para mitigar esa relación.


### 5. Entrenamiento de modelos

In [7]:
# Preparar los datos en datos de entrenamiento y datos de prueba
features = data.drop("is_ultra", axis=1)
target = data["is_ultra"]
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.33, random_state=42)
# Nota: La estandarización se realiza después de dividir los datos para evitar la fuga de datos (data leakage)
scaler = StandardScaler()
scaler.fit(x_train)
x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [8]:
# Modelo Logistico
modelo_logistico = LogisticRegression(class_weight='balanced', random_state=42)
modelo_logistico.fit(x_train_scaled, y_train)
y_predict_train = modelo_logistico.predict(x_train_scaled) # Predicciones en el conjunto de entrenamiento
y_predict_test = modelo_logistico.predict(x_test_scaled) # Predicciones en el conjunto de prueba
score_train = accuracy_score(y_train, y_predict_train)
score_test = accuracy_score(y_test, y_predict_test)

precision_train = precision_score(y_train, y_predict_train)
precision_test = precision_score(y_test, y_predict_test)

f1_train = f1_score(y_train, y_predict_train)
f1_test = f1_score(y_test, y_predict_test)

result = {
    "logistic_model": {
        "score_train_logistico": score_train,
        "score_test_logistico": score_test,
        "precision_train_logistico": precision_train,
        "precision_test_logistico": precision_test,
        "f1_train_logistico": f1_train,
        "f1_test_logistico": f1_test
    }
}

In [9]:
# Modelo Decision Tree 
# NOTA: Decision Tree no necesita estandarización, por lo que se utiliza x_train y x_test sin escalar
best_score = 0
best_f1_score = 0
for depth in range(1,10):
    modelo_decision_tree = DecisionTreeClassifier(class_weight='balanced', random_state=42, max_depth=depth)
    modelo_decision_tree.fit(x_train, y_train)
    
    y_predict_train = modelo_decision_tree.predict(x_train)
    y_predict_test = modelo_decision_tree.predict(x_test)
    
    score_train = accuracy_score(y_train, y_predict_train)
    score_test = accuracy_score(y_test, y_predict_test)

    f1_train = f1_score(y_train, y_predict_train)
    f1_test = f1_score(y_test, y_predict_test)
        
    if abs(f1_train - f1_test) < 0.1 and abs(score_train - score_test) < 0.1 and score_test > best_score:
        best_score_train = score_train
        best_score_test = score_test
        best_f1_train = f1_train
        best_f1_score_test = f1_test
        best_depth = depth

result["decision_tree_model"] = {
    "score_train": best_score_train,
    "score_test": best_score_test, 
    "f1_train": best_f1_train,
    "f1_test": best_f1_score_test,
    "best_depth": best_depth
}

In [ ]:
# Modelo Random Forest
# Nota: Random Forest no necesita estandarización, por lo que se utiliza x_train y x_test sin escalar
best_score = 0
best_f1_score = 0
for est in [200, 250, 300]:
    for depth in range(3,15, 1):
        for leaf in range(3, 7,1):
            modelo_random_forest = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=est, max_depth=depth,
                                                        min_samples_split=5, min_samples_leaf=leaf)
            modelo_random_forest.fit(x_train, y_train)
            
            y_predict_train = modelo_random_forest.predict(x_train)
            y_predict_test = modelo_random_forest.predict(x_test)
            
            score_train = accuracy_score(y_train, y_predict_train)
            score_test = accuracy_score(y_test, y_predict_test)
            
            f1_train = f1_score(y_train, y_predict_train)
            f1_test = f1_score(y_test, y_predict_test)
            
            if abs(f1_train - f1_test) < 0.1 and abs(score_train - score_test) < 0.1 and score_test > best_score:
                best_score_train = score_train
                best_score_test = score_test
                best_f1_train = f1_train
                best_f1_score_test = f1_test
                best_est = est
                best_depth = depth
                best_leaf = leaf
            
result["random_forest_model"] = {
    "score_train": best_score_train,
    "score_test": best_score_test, 
    "f1_train": best_f1_train,
    "f1_test": best_f1_score_test,
    "best_estimator": best_est,
    "best_depth": best_depth,
    "best_leaf": best_leaf

}

### 6. Mostrar resultados

In [11]:
for key, value in result.items():
    print(f"*****{key.replace('_', ' ').title()}*****")
    for metric_key, metric_value in value.items():
        print(f"{metric_key.replace('_', ' ').title()}: {metric_value:.2f}")
    print("\n")

*****Logistic Model*****
Score Train Logistico: 0.64
Score Test Logistico: 0.62
Precision Train Logistico: 0.44
Precision Test Logistico: 0.42
F1 Train Logistico: 0.51
F1 Test Logistico: 0.49


*****Decision Tree Model*****
Score Train: 0.83
Score Test: 0.79
F1 Train: 0.70
F1 Test: 0.62
Best Depth: 6.00


*****Random Forest Model*****
Score Train: 0.86
Score Test: 0.80
F1 Train: 0.75
F1 Test: 0.65
Best Estimator: 300.00
Best Depth: 10.00
Best Leaf: 6.00




### 7. Prueba de cordura

In [ ]:
# 7.1. Hacer dummy baseline

In [ ]:
# 7.2. Hacer la mezcla de los datos

In [ ]:
# 7.3. Crear un overfitting intencional

### 8. Conclusiones

1. Decision Tree fue un modelo que ofreció un buen resultado (accuracy = 0.79, F1=0.62) a un MUY BAJO costo computacional, la operación tardó 0.1 segundos en realizarse. Además, este modelo mostró un ajuste ideal, sin overfitting ni underfitting. Vale la pena mencionar que la prueba F1 tiene un resultado moderado, es decir que el modelo no siempre acierta el tipo de usuario.

2. El modelo Random Forest, ofreció los mejores resultados con los datos de entrenamiento (accuracy = 0.86 y F1 = 0.75); sin embargo, al utilizar los datos de prueba los resultados fueron muy similares a los obtenidos con el Decision Tree (accuracy = 0.8, F1 = 0.65); sin embargo, el costo computacional fue casi 230 veces mayor que el costo computacional del modelo Decision Tree.

3. El peor modelo fue el modelo Logistico con un accuracy de 0.64 y un F1 de 0.49; indicando que el modelo no se ajustó bien a los datos y las predicciones de los planes de los usuarios fue débil.

4. Con respecto al ajuste de los hiperparámetros de los modelos; el modelo Decision Tree indicó que con una profundidad de 5 se obtenía el mejor resultado. Mientras que en el modelo de Random Forest se recorrieron diferentes profunidades, estimadores y número mínimo de hojas; de esta forma se encontraron los hiperparámetros mostrados en el punto dos de estas conclusiones.

5. Las comparaciones de los puntajes de las predicciones de los datos de entrenamiento vs los puntajes de las predcciones de los datos de prueba mostraron que no había underfitting ni overfitting en los modelos.

6. La prueba de cordura sólo se le realizó al modelo de Random Forest
